# 可靠邮件与统一数据表
三个公开来源统一存放，来源、标签依据、原文哈希与用途仍可追溯。这里只看聚合质量和合成解析示例，不打印真实邮件全文。

In [1]:
from pathlib import Path
import os,json
if not Path('configs').exists() and Path('../configs').exists():os.chdir('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from spam_analysis.analysis.results import source_records,result_table,error_frame
from spam_analysis.experiments.runtime import ExperimentSession
plt.rcParams['font.sans-serif']=['Microsoft YaHei']
plt.rcParams['axes.unicode_minus']=False
engine=ExperimentSession()
print('结果版本：',engine.run_id)


结果版本： course-20261016


In [2]:
frame=source_records('combined')
assert frame.doc_id.is_unique
assert frame.groupby('template_group_id').split.nunique().max()==1
display(pd.crosstab(frame.source_id,frame.label))
display(pd.crosstab(frame.source_id,frame.split))

label,0,1
source_id,,
enron,18003,25179
spamassassin,4108,1805
trec06c,21766,42854


split,calibrate,policy,test,train,tune
source_id,,,,,
enron,4281,4363,8884,21406,4248
spamassassin,583,585,1225,2933,587
trec06c,6579,6239,9711,36158,5933


In [3]:
quality=frame.groupby('source_id').agg(邮件数=('doc_id','size'),正文为空=('body_missing','sum'),含损坏字符=('replacement_ratio',lambda s:s.gt(0).sum()))
display(quality)
display(pd.crosstab(frame.script_profile,frame.label))

,邮件数,正文为空,含损坏字符
source_id,,,
enron,43182,632,273
spamassassin,5913,28,89
trec06c,64620,335,10782


label,0,1
script_profile,,
中拉混合,843,6168
中文字符为主,20685,35710
其他 / 难判,70,415
拉丁字母为主,22274,26555
空正文,5,990


文字类型来自正文比例规则，不是人工语言标注。原 TREC06C 内容与划分冻结，新来源先排除跨来源重复，再按模板组划分。来源可靠不等于全部标注绝无错误。

In [4]:
from spam_analysis.parsing import parse_email
raw=b'Content-Type: text/plain; charset=gb2312\nContent-Transfer-Encoding: base64\n\n'+'课程通知，请按时参加。'.encode('gbk')
parsed=parse_email(raw)
display({k:parsed[k] for k in ['body_clean','used_charset','fallback_reason']})
assert parsed['body_clean']=='课程通知,请按时参加。'

{'body_clean': '课程通知,请按时参加。',
 'used_charset': 'gb18030',
 'fallback_reason': 'invalid_base64_kept_raw'}

原始字节与声明可能不一致。验证传输编码后再解码，保留无法恢复的损坏标记；不通过猜字补齐内容。